#  load data

In [ ]:
from datasets import load_dataset

datasets = ["narrativeqa", "qasper", "multifieldqa_en", "multifieldqa_zh", "hotpotqa", "2wikimqa", "musique",
            "dureader", "gov_report", "qmsum", "multi_news", "vcsum", "trec", "triviaqa", "samsum", "lsht",
            "passage_count", "passage_retrieval_en", "passage_retrieval_zh", "lcc", "repobench-p"]


data = load_dataset('THUDM/LongBench', "musique")


 # LangchainDocument

In [ ]:
from tqdm import tqdm
from langchain.docstore.document import Document

RAW_KNOWLEDGE_BASE = [
    Document(
        page_content=doc["context"],
        metadata={
            "question": doc["input"],
            "short_answer": doc["answers"]
        }
    )
    for doc in tqdm(data["test"])  # Iterate over the correct split of the dataset
]

## chunking

In [ ]:
import torch
from langchain.text_splitter import CharacterTextSplitter

from langchain.docstore.document import Document as LangchainDocument

def chunk_documents(documents, chunk_size=700, chunk_overlap=0):
    """
    Chunk documents while preserving their original metadata

    Args:
    - documents (list): List of LangchainDocuments to be chunked
    - chunk_size (int): Maximum number of characters in each chunk
    - chunk_overlap (int): Number of characters to overlap between chunks

    Returns:
    - List of chunked LangchainDocuments
    """
    # Initialize the text splitter
    text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    # List to store all chunked documents
    chunked_documents = []

    # Iterate through each original document
    for doc in tqdm(documents):
        # Split the document's content
        text_chunks = text_splitter.split_text(doc.page_content)

        # Create new documents for each chunk, preserving original metadata
        for i, chunk in enumerate(text_chunks):
            # Create a copy of the original metadata
            chunk_metadata = doc.metadata.copy()

            # Add chunk-specific information
            chunk_metadata['chunk_id'] = i
            chunk_metadata['total_chunks'] = len(text_chunks)

            # Create a new document with the chunk and updated metadata
            chunked_doc = LangchainDocument(
                page_content=chunk,
                metadata=chunk_metadata
            )

            chunked_documents.append(chunked_doc)

    return chunked_documents



In [ ]:
# Chunk the documents
docs_processed = chunk_documents(RAW_KNOWLEDGE_BASE)
len(docs_processed)